**Goal: explain choice of data and project goal**

I chose the following dataset: 

@ *https://huggingface.co/datasets/ccdv/arxiv-summarization*

It alligned with what I want to accomplish which is title generation for academic papers. The dataset contains papers from multiple fields of STEM allowing me to focus on either one field or multiple for my model. It also helps that its pretty organized

**Project data summary: data index, format, size in GB, # rows, statistical characteristics of data**

Formatting: *Apache Parquet*

Size: *7.26 GB*

Number of Rows: *431,826*

Characteristics: *Word Counts, Character Counts*

**Data Ingestion**

In [3]:
import pandas as pd
from datasets import load_dataset

c:\Users\widin\anaconda3\envs\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
ds = load_dataset("ccdv/arxiv-summarization", "section")

c:\Users\widin\anaconda3\envs\nlp\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\widin\.cache\huggingface\hub\datasets--ccdv--arxiv-summarization. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 6440/6440 [00:00<00:00, 8394.77 examples/s] 


In [5]:
print(ds)
print(ds["train"].column_names)
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['article', 'abstract'],
        num_rows: 203037
    })
    validation: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6436
    })
    test: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6440
    })
})
['article', 'abstract']
{'article': 'additive models @xcite provide an important family of models for semiparametric regression or classification . some reasons for the success of additive models are their increased flexibility when compared to linear or generalized linear models and their increased interpretability when compared to fully nonparametric models . \n it is well - known that good estimators in additive models are in general less prone to the curse of high dimensionality than good estimators in fully nonparametric models . \n many examples of such estimators belong to the large class of regularized kernel based methods over a reproducing kernel hilbert space @xmath0 , see e.

In [6]:
dataset_doc = load_dataset("ccdv/arxiv-summarization", "document")

Generating test split: 100%|██████████| 6440/6440 [00:00<00:00, 9543.23 examples/s] 


In [7]:
print(dataset_doc["train"].column_names)
print(dataset_doc["train"][0].keys())

['article', 'abstract']
dict_keys(['article', 'abstract'])


This dataset has plentiful to it but only contains the abstract and article. No title and no id to trace back the article to get a title. Another concern that presented itself after viewing this dataset is that if I decide to train on both the abstract portion and the article of a paper, it will take a lot of computational power that I don't have. So I decided to pivot over and started looking for datasets that contained at least the title of the paper and the abstract.


I found the following:

@ *https://huggingface.co/datasets/librarian-bots/arxiv-metadata-snapshot*

In [8]:
dataset = load_dataset("librarian-bots/arxiv-metadata-snapshot")

In [6]:
print(dataset["train"].column_names)
print(dataset["train"][0])

['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed']
{'id': '0809.2509', 'submitter': 'Vitaly Velizhanin', 'authors': 'V.N. Velizhanin (St. Petersburg, INP)', 'title': 'Three-loop renormalization of the N=1, N=2, N=4 supersymmetric Yang-Mills theories', 'comments': '6 pages, mismatch for N=2 SYM theory corrected, references updated', 'journal-ref': None, 'doi': '10.1016/j.nuclphysb.2009.03.017', 'report-no': None, 'categories': 'hep-th', 'license': 'http://arxiv.org/licenses/nonexclusive-distrib/1.0/', 'abstract': 'We calculate the renormalization constants of the N=1, N=2, N=4 supersymmetric Yang-Mills theories in an arbitrary covariant gauge in the dimensional reduction scheme up to three loops. We have found, that the beta-functions for N=1 and N=4 SYM theories are the same from the different triple vertices. This means that the dimensional reduction scheme works corre

It seems the second one is more up to date and has more content to it. Also as another plus it has a categories column allowing me to pick and choose on training CS papers for CS title predictions or Bio for Bio titles.

So I am going to pivot over to this dataset instead

New Dataset:

Formatting: *Apache Parquet*

Size: *2.77 GB*

Number of Rows: *2,975,294*

Characteristics: *id, submitter, authors, title, comments, journal-ref, doi, report-no, categories, license, abstract, versions, update_date, authors_parsed*

Not only does this dataset have what im looking for (title, abstract), it also contains a bunch of other characteristics, more notably the id. This will allow me to trace back the original document and scrape for more information if I decide to test with more than just the abstract portions. But for now I'm sticking with the title and abstract.

In [9]:
train_data = dataset["train"]
print("Dataset loaded successfully.")
print(f"  Total rows : {train_data.num_rows:,}")
print(f"  Columns    : {train_data.column_names}")

Dataset loaded successfully.
  Total rows : 2,975,294
  Columns    : ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed']


We can omit many of the columns

In [10]:
train_data = train_data.select_columns(["id", "title", "abstract", "categories", "update_date"])

print(f"\nAfter column selection: {train_data.column_names}")


After column selection: ['id', 'title', 'abstract', 'categories', 'update_date']


**Data Normalizaion:** *Sandbox*

Sentence Tokenizers
Word Tokenizers
Token Normalize

In [ ]:
import re